# HTA-MAC Phase 2D QoS-constrained training

This notebook runs the frozen **development-only** confirmation: equivariant set Branching Dueling C51, TTL/cap-aware observations, budget 12, three optimizer lineages, and adaptive delivery/staleness/fairness constraints. It never trains or selects on held-out Phase 3 seeds 3100-3104. A successful run authorizes later held-out evaluation; it is not itself a publication result.


In [ ]:
# Frozen user settings. Change only paths/download preference.
BUNDLE_PATH = '/content/HTA_MAC_Phase2D_QoS_Training_Bundle_20260804.zip'
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/HTA_MAC_Phase2D_QoS_Training_20260804'
SEEDS = [2299, 3299, 4299]
EPISODES = 500
DOWNLOAD_RESULTS_WHEN_COMPLETE = True
EXPECTED_BUNDLE_SHA256 = '0be9a01716800a0f79505aa6ba2089573ef496e20c718ab54a3b38fe33278a48'
assert SEEDS == [2299, 3299, 4299]
assert EPISODES == 500


In [ ]:
# GPU, Drive, bundle discovery, checksum, safe extraction, and per-file manifest verification.
import glob, hashlib, json, os, shutil, subprocess, sys, zipfile
from pathlib import Path, PurePosixPath
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime > Change runtime type > GPU before training.')
print('GPU:', torch.cuda.get_device_name(0), '| Torch:', torch.__version__)
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Colab; Drive mount skipped.')
bundle = Path(BUNDLE_PATH)
if not bundle.is_file():
    candidates = [Path(p) for p in glob.glob('/content/*Phase2D*QoS*Bundle*.zip')]
    if len(candidates) != 1:
        raise FileNotFoundError(f'Expected one uploaded bundle, found {candidates}')
    bundle = candidates[0]
digest = hashlib.sha256(bundle.read_bytes()).hexdigest()
assert digest == EXPECTED_BUNDLE_SHA256, (digest, EXPECTED_BUNDLE_SHA256)
extract_root = Path('/content/hta_mac_phase2d_qos')
if extract_root.exists():
    shutil.rmtree(extract_root)
extract_root.mkdir(parents=True)
with zipfile.ZipFile(bundle) as archive:
    for member in archive.infolist():
        normalized = member.filename.replace('\\', '/')
        path = PurePosixPath(normalized)
        if path.is_absolute() or '..' in path.parts:
            raise RuntimeError(f'Unsafe ZIP member: {member.filename}')
        target = extract_root.joinpath(*path.parts)
        if member.is_dir() or normalized.endswith('/'):
            target.mkdir(parents=True, exist_ok=True)
            continue
        target.parent.mkdir(parents=True, exist_ok=True)
        with archive.open(member, 'r') as source, target.open('wb') as destination:
            shutil.copyfileobj(source, destination)
stage2 = extract_root / 'stage2'
repo = stage2 / 'hta-mac'
upstream = stage2 / 'final_repo'
manifest = json.loads((stage2 / 'COLAB_PHASE2D_QOS_MANIFEST.json').read_text(encoding='utf-8-sig'))
assert manifest['optimizer_seeds'] == SEEDS and manifest['episodes'] == EPISODES
assert set(manifest['development_seeds']).isdisjoint(manifest['prohibited_held_out_seeds'])
for entry in manifest['files']:
    target = stage2 / entry['path']
    assert target.is_file() and target.stat().st_size == entry['bytes'], entry['path']
    assert hashlib.sha256(target.read_bytes()).hexdigest() == entry['sha256'], entry['path']
print('Verified bundle:', bundle.name, '| files:', len(manifest['files']), '| SHA256:', digest)


In [ ]:
# Install declared runtime/test dependencies.
packages = ['gymnasium>=0.29,<2', 'torch-geometric>=2.4,<3', 'numpy>=1.24', 'scipy>=1.10', 'pyyaml>=6', 'pytest>=7']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)
print('Dependencies installed.')


In [ ]:
# Strict preflight: compile all source and run the complete validation suite.
subprocess.run([sys.executable, '-m', 'compileall', '-q', str(repo), str(upstream)], check=True)
subprocess.run([sys.executable, '-B', '-m', 'pytest', 'validation', '-q', '-p', 'no:cacheprovider'], cwd=repo, check=True)
print('Preflight passed.')


## Train the three registered lineages

Each lineage is a fresh 500-episode development run. Completed, gate-passing lineage folders in Drive are reused; an interrupted or failed lineage restarts cleanly. Every completed checkpoint is subjected to the Phase 2D equivariance, action-feasibility, and categorical-support audit before backup.


In [ ]:
drive_root = Path(DRIVE_OUTPUT_DIR)
drive_root.mkdir(parents=True, exist_ok=True)
local_phase2 = repo / 'outputs' / 'phase2'
local_phase2.mkdir(parents=True, exist_ok=True)
env = dict(os.environ, PYTHONUNBUFFERED='1')
run_names = []
for seed in SEEDS:
    run_name = f'phase2d_qos_equivariant_500ep_seed{seed}'
    run_names.append(run_name)
    local_run = local_phase2 / run_name
    drive_run = drive_root / run_name
    complete = False
    if (drive_run / 'summary.json').is_file() and (drive_run / 'phase2d_foundation_audit.json').is_file():
        summary = json.loads((drive_run / 'summary.json').read_text())
        audit = json.loads((drive_run / 'phase2d_foundation_audit.json').read_text())
        complete = bool(summary.get('phase2_curriculum_gate_pass') and audit.get('status') == 'gate_pass')
    if complete:
        print('Reusing verified Drive result:', run_name)
        if local_run.exists(): shutil.rmtree(local_run)
        shutil.copytree(drive_run, local_run)
        continue
    if local_run.exists(): shutil.rmtree(local_run)
    command = [sys.executable, '-B', 'experiments/train_phase2_dynamic_curriculum.py',
        '--episodes', str(EPISODES), '--max-steps', '300',
        '--development-seeds', '2300,2301,2302,2303,2304',
        '--optimizer-seed', str(seed), '--run-name', run_name, '--device', 'cuda',
        '--learn-every', '4', '--reward-scale-config', 'config/phase2c_return_scale.json',
        '--qos-constraint-config', 'config/phase2d_qos_constraints.json',
        '--projection-budget', '12', '--architecture', 'equivariant_set_branching',
        '--stability-interval', '50', '--stability-tail-episodes', '100',
        '--learning-rate', '1e-5', '--normalize-input-blocks',
        '--trajectory-loss-weight', '1.0', '--concavity-loss-weight', '0.1', '--precision', 'fp32']
    print('TRAINING:', run_name)
    subprocess.run(command, cwd=repo, env=env, check=True)
    checkpoint = local_run / 'branching_c51.pt'
    audit_path = local_run / 'phase2d_foundation_audit.json'
    subprocess.run([sys.executable, '-B', 'experiments/audit_phase2d_foundation.py', str(checkpoint), '--output', str(audit_path)], cwd=repo, check=True)
    summary = json.loads((local_run / 'summary.json').read_text())
    audit = json.loads(audit_path.read_text())
    assert summary['phase2_curriculum_gate_pass'] and audit['status'] == 'gate_pass'
    if drive_run.exists(): shutil.rmtree(drive_run)
    shutil.copytree(local_run, drive_run)
    print('Backed up verified lineage to Drive:', drive_run)


In [ ]:
# Freeze development selection without touching held-out seeds.
selection_rows = []
for run_name, seed in zip(run_names, SEEDS):
    run_dir = local_phase2 / run_name
    summary = json.loads((run_dir / 'summary.json').read_text())
    audit = json.loads((run_dir / 'phase2d_foundation_audit.json').read_text())
    episodes = [json.loads(line) for line in (run_dir / 'episodes.jsonl').read_text().splitlines() if line.strip()]
    tail = episodes[-50:]
    mean_violation = sum(sum(row['qos_constraint']['positive_violations'].values()) for row in tail) / len(tail)
    selection_rows.append({'seed': seed, 'run_name': run_name, 'mean_last50_positive_qos_violation': mean_violation,
        'mean_throughput': summary['greedy_evaluation']['mean_throughput'],
        'mean_queue_fairness': summary['greedy_evaluation']['mean_queue_fairness'],
        'curriculum_gate_pass': summary['phase2_curriculum_gate_pass'], 'foundation_audit': audit['status'],
        'checkpoint_sha256': hashlib.sha256((run_dir / 'branching_c51.pt').read_bytes()).hexdigest()})
assert all(row['curriculum_gate_pass'] and row['foundation_audit'] == 'gate_pass' for row in selection_rows)
selection_rows.sort(key=lambda row: (row['mean_last50_positive_qos_violation'], -row['mean_queue_fairness'], -row['mean_throughput']))
selection = {'status': 'development_selection_frozen', 'held_out_seeds_used': False, 'ranking_rule': 'minimum last-50 cumulative positive QoS violation; fairness then throughput tie-break',
    'selected_seed': selection_rows[0]['seed'], 'selected_run_name': selection_rows[0]['run_name'], 'lineages': selection_rows,
    'next_step': 'Run one locked held-out Phase 3 evaluation on seeds 3100-3104; do not retrain after viewing it.'}
selection_path = drive_root / 'DEVELOPMENT_SELECTION.json'
selection_path.write_text(json.dumps(selection, indent=2))
print(json.dumps(selection, indent=2))


In [ ]:
# Create one result archive in Drive and optionally download it.
archive_base = drive_root.parent / 'HTA_MAC_Phase2D_QoS_Trained_Results_20260804'
archive = Path(shutil.make_archive(str(archive_base), 'zip', root_dir=drive_root))
archive_hash = hashlib.sha256(archive.read_bytes()).hexdigest()
archive.with_suffix('.zip.sha256').write_text(f'{archive_hash}  {archive.name}\n')
print('RESULTS:', archive, '| SHA256:', archive_hash)
if DOWNLOAD_RESULTS_WHEN_COMPLETE:
    try:
        from google.colab import files
        files.download(str(archive))
        files.download(str(archive.with_suffix('.zip.sha256')))
    except ImportError:
        print('Download skipped outside Colab.')
